# ANRF AISEHack 2.0 — Polymer Property Prediction v7
**v4/v6 public LB: 0.896 | OOF: 0.9117  (Tg: 0.9082, Egc: 0.9151)**

**v7 changes:**
- **Optuna hyperparameter tuning** for XGB per target (25 trials × 3-fold each)
- ET dropped — got 0.000 blend weight in v6
- LGBM + XGB_tuned, 5 seeds × 10 folds as final ensemble
- XGB carries 76–88% of blend weight; tuning it is the highest-leverage remaining move


In [ ]:
!pip install rdkit optuna -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import glob

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator, RDKFingerprint

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

In [ ]:
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)[0]

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'Tg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

## Feature Engineering

**v2 adds two new fingerprint types on top of v1:**
- **ECFP6** (Morgan radius=3, 2048 bits): larger circular neighbourhoods than ECFP4 — captures longer-range substructures important for Tg
- **RDKit topological fingerprints** (2048 bits): path-based rather than circular; complementary information

Total raw features: ~4,500 (up from ~2,400). Variance filtering still applied.

In [ ]:
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))

    return pd.concat([
        pd.DataFrame(rdkit_rows,  columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows,  columns=[f'ecfp4_{i}'  for i in range(2048)]),
        pd.DataFrame(ecfp6_rows,  columns=[f'ecfp6_{i}'  for i in range(2048)]),
        pd.DataFrame(rdk_rows,    columns=[f'rdkfp_{i}'  for i in range(2048)]),
        pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)]),
    ], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    X_imp    = imputer.fit_transform(X)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

In [ ]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw       = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw      = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')
print('Preprocessing ...')

X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

## Step 1 — Optuna: Tune XGB Hyperparameters Per Target

XGB carries 76–88% of the final blend weight (from v4/v6 analysis), so tuning it has direct impact on the submission.

**Search strategy:**
- 25 trials per target (Tg and Egc separately)
- Each trial: 3-fold CV with early stopping — fast enough for 25 trials within budget
- Learning rate fixed at 0.01 — early stopping controls `n_estimators` implicitly
- Search space: `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`

After search: best params are used for all 5 seeds × 10 folds in the final ensemble.

In [ ]:
SEEDS = [42, 7, 123, 17, 99]
N_TRIALS = 25   # per target — ~3-4 min per trial, ~1.5h total for both targets

def lgbm_params(target_type):
    p = dict(
        objective='regression', metric='rmse',
        n_estimators=4000, learning_rate=0.01,
        num_leaves=127, max_depth=-1,
        min_child_samples=15,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, verbose=-1,
    )
    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20
    return p


def xgb_base_params(target_type):
    """Fixed params shared across all XGB runs (Optuna and final)."""
    p = dict(
        objective='reg:squarederror',
        n_estimators=4000,
        learning_rate=0.01,
        tree_method='hist',
        early_stopping_rounds=200,
        n_jobs=-1,
    )
    return p


print('Base params defined.')

In [ ]:
def make_xgb_objective(X_train, y_train, target_type):
    """Returns an Optuna objective that does 3-fold CV for one XGB config."""
    def objective(trial):
        params = xgb_base_params(target_type)
        params.update({
            'max_depth'        : trial.suggest_int('max_depth', 3, 8),
            'min_child_weight' : trial.suggest_int('min_child_weight', 1, 20),
            'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.2, 0.8),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'random_state'     : 42,
        })

        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        r2_scores = []
        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            m = xgb.XGBRegressor(**params)
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            r2_scores.append(r2_score(y_val, m.predict(X_val)))
        return np.mean(r2_scores)
    return objective


print('Tuning XGB for Tg ...')
study_tg = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_tg.optimize(make_xgb_objective(X_tg, y_tg, 'tg'),
                  n_trials=N_TRIALS, show_progress_bar=True)

best_tg = study_tg.best_params
print(f'Best Tg  XGB params: {best_tg}')
print(f'Best Tg  3-fold R²:  {study_tg.best_value:.4f}')

print('\nTuning XGB for Egc ...')
study_egc = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_egc.optimize(make_xgb_objective(X_egc, y_egc, 'egc'),
                   n_trials=N_TRIALS, show_progress_bar=True)

best_egc = study_egc.best_params
print(f'Best Egc XGB params: {best_egc}')
print(f'Best Egc 3-fold R²:  {study_egc.best_value:.4f}')


In [ ]:
def train_ensemble(X_train, y_train, X_test, target_type, seed,
                   xgb_tuned_params=None, n_splits=10):
    """
    10-fold CV ensemble of LGBM + XGB (with tuned params) for one seed.
    Returns (oof_l, oof_x), (test_l, test_x).
    """
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test  = np.asarray(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_lgbm  = np.zeros(len(X_train))
    oof_xgb   = np.zeros(len(X_train))
    test_lgbm = np.zeros(len(X_test))
    test_xgb  = np.zeros(len(X_test))

    lp = lgbm_params(target_type)
    lp['random_state'] = seed

    xp = xgb_base_params(target_type)
    if xgb_tuned_params is not None:
        xp.update(xgb_tuned_params)   # inject Optuna-found params
    xp['random_state'] = seed

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        m_lgbm = lgb.LGBMRegressor(**lp)
        m_lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(200, verbose=False),
                              lgb.log_evaluation(period=0)])
        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm        += m_lgbm.predict(X_test) / n_splits

        m_xgb = xgb.XGBRegressor(**xp)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb        += m_xgb.predict(X_test) / n_splits

        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        print(f'    fold {fold:02d} | LGBM={r2_l:.4f}  XGB={r2_x:.4f}')

    print(f'  OOF  LGBM={r2_score(y_train, oof_lgbm):.4f}  '
          f'XGB={r2_score(y_train, oof_xgb):.4f}')

    return (oof_lgbm, oof_xgb), (test_lgbm, test_xgb)


print('Training function defined.')

In [ ]:
print('=' * 60)
print('  Tg  (glass transition temperature)')
print('=' * 60)

tg_oof_parts_all  = []
tg_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(
        X_tg, y_tg, X_tg_test, 'tg', seed=seed,
        xgb_tuned_params=best_tg
    )
    tg_oof_parts_all.append(oof_parts)
    tg_test_parts_all.append(test_parts)

oof_l_tg  = np.mean([p[0] for p in tg_oof_parts_all],  axis=0)
oof_x_tg  = np.mean([p[1] for p in tg_oof_parts_all],  axis=0)
test_l_tg = np.mean([p[0] for p in tg_test_parts_all], axis=0)
test_x_tg = np.mean([p[1] for p in tg_test_parts_all], axis=0)

oof_tg_parts  = (oof_l_tg, oof_x_tg)
test_tg_parts = (test_l_tg, test_x_tg)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_tg, oof_l_tg):.4f}  '
      f'XGB={r2_score(y_tg, oof_x_tg):.4f}')


In [ ]:
print('=' * 60)
print('  Egc  (chain band gap)')
print('=' * 60)

egc_oof_parts_all  = []
egc_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(
        X_egc, y_egc, X_egc_test, 'egc', seed=seed,
        xgb_tuned_params=best_egc
    )
    egc_oof_parts_all.append(oof_parts)
    egc_test_parts_all.append(test_parts)

oof_l_egc  = np.mean([p[0] for p in egc_oof_parts_all],  axis=0)
oof_x_egc  = np.mean([p[1] for p in egc_oof_parts_all],  axis=0)
test_l_egc = np.mean([p[0] for p in egc_test_parts_all], axis=0)
test_x_egc = np.mean([p[1] for p in egc_test_parts_all], axis=0)

oof_egc_parts  = (oof_l_egc, oof_x_egc)
test_egc_parts = (test_l_egc, test_x_egc)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_egc, oof_l_egc):.4f}  '
      f'XGB={r2_score(y_egc, oof_x_egc):.4f}')


## Optimise Blend Weights on OOF

Find the weight vector `[w_lgbm, w_xgb, w_cat]` that maximises OOF R² per target via constrained optimisation (`sum=1`, all weights ≥0).  
These weights are then applied to the **test predictions** — not just used diagnostically.  
The individual test arrays (`test_lgbm`, `test_xgb`, `test_cat`) are returned from `train_ensemble` for exactly this purpose.

In [ ]:
from scipy.optimize import minimize

def find_best_weights(oof_parts, y_true):
    oof_l, oof_x = oof_parts
    stack = np.column_stack([oof_l, oof_x])
    def neg_r2(w):
        return -r2_score(y_true, stack @ w)
    res = minimize(
        neg_r2, x0=[0.5, 0.5], method='SLSQP',
        bounds=[(0, 1)] * 2,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
    )
    return res.x

w_tg  = find_best_weights(oof_tg_parts,  y_tg)
w_egc = find_best_weights(oof_egc_parts, y_egc)

print(f'Optimal Tg  weights — LGBM: {w_tg[0]:.3f}  XGB: {w_tg[1]:.3f}')
print(f'Optimal Egc weights — LGBM: {w_egc[0]:.3f}  XGB: {w_egc[1]:.3f}')

oof_l_tg,  oof_x_tg  = oof_tg_parts
oof_l_egc, oof_x_egc = oof_egc_parts
test_l_tg,  test_x_tg  = test_tg_parts
test_l_egc, test_x_egc = test_egc_parts

oof_tg_opt  = w_tg[0]*oof_l_tg  + w_tg[1]*oof_x_tg
oof_egc_opt = w_egc[0]*oof_l_egc + w_egc[1]*oof_x_egc

r2_tg  = r2_score(y_tg,  oof_tg_opt)
r2_egc = r2_score(y_egc, oof_egc_opt)

print(f'\nOOF R² Tg  : {r2_tg:.4f}   (v6: 0.9082)')
print(f'OOF R² Egc : {r2_egc:.4f}   (v6: 0.9151)')
print(f'Mean OOF R²: {(r2_tg + r2_egc)/2:.4f}   (v6: 0.9117)')

pred_tg  = w_tg[0]*test_l_tg  + w_tg[1]*test_x_tg
pred_egc = w_egc[0]*test_l_egc + w_egc[1]*test_x_egc

print('\nTest predictions updated with optimised weights.')


In [ ]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission.head(10))
print('\nsubmission.csv saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_tg, oof_tg_opt, alpha=0.25, s=8)
lo, hi = min(y_tg.min(), oof_tg_opt.min()), max(y_tg.max(), oof_tg_opt.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('True Tg (°C)', fontsize=12)
axes[0].set_ylabel('Pred Tg (°C)', fontsize=12)
axes[0].set_title(f'Tg OOF  R² = {r2_tg:.4f}', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_egc, oof_egc_opt, alpha=0.25, s=8, color='darkorange')
lo, hi = min(y_egc.min(), oof_egc_opt.min()), max(y_egc.max(), oof_egc_opt.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[1].set_xlabel('True Egc (eV)', fontsize=12)
axes[1].set_ylabel('Pred Egc (eV)', fontsize=12)
axes[1].set_title(f'Egc OOF  R² = {r2_egc:.4f}', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'v7 OOF Predicted vs Actual  |  Mean R² = {(r2_tg+r2_egc)/2:.4f}   (v6: 0.896 LB)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/oof_scatter_v7.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')